# ST-GCN Training — NTU RGB+D 60 XSub

**Çalıştırmadan önce:**
1. `processed_xsub.npz` dosyasını Google Drive'a yükle → `MyDrive/stgcn/processed_xsub.npz`
2. Colab runtime'ı **A100 GPU** olarak seç
3. Hücreleri sırayla çalıştır

## 0. Drive Bağla & Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from collections import defaultdict
import time

# GPU kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Çıktı klasörü
OUT_DIR = '/content/drive/MyDrive/stgcn/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

DATA_PATH = '/content/drive/MyDrive/stgcn/processed_xsub.npz'
print("Data path:", DATA_PATH)
print("Exists:", os.path.exists(DATA_PATH))

## 1. Dataset

In [ ]:
NTU60_CLASSES = [
    'drink water', 'eat meal/snack', 'brushing teeth', 'brushing hair', 'drop',
    'pickup', 'throw', 'sitting down', 'standing up', 'clapping',
    'reading', 'writing', 'tear up paper', 'wear jacket', 'take off jacket',
    'wear a shoe', 'take off a shoe', 'wear on glasses', 'take off glasses', 'put on a hat/cap',
    'take off a hat/cap', 'cheer up', 'hand waving', 'kicking something', 'reach into pocket',
    'hopping (one foot jumping)', 'jump up', 'make a phone call/answer phone',
    'playing with phone/tablet', 'typing on a keyboard',
    'pointing to something with finger', 'taking a selfie', 'check time (from watch)',
    'rub two hands together', 'nod head/bow', 'shake head', 'wipe face', 'salute',
    'put the palms together', 'cross hands in front', 'sneeze/cough', 'staggering',
    'falling', 'touch head (headache)', 'touch chest (stomachache/heart pain)',
    'touch back (backache)', 'touch neck (neckache)', 'nausea or vomiting condition',
    'use a fan (with hand or paper)/feeling warm', 'punching/slapping other person',
    'kicking other person', 'pushing other person', 'pat on back of other person',
    'point finger at the other person', 'hugging other person',
    'giving something to other person', 'touch other persons pocket',
    'handshaking', 'walking towards each other', 'walking apart from each other'
]

class NTUDataset(Dataset):
    def __init__(self, X, y, augment=False):
        # X: (N, M, T, V, C)  →  model (N, C, T, V, M) formatına geçilecek
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]  # (M, T, V, C)
        if self.augment:
            x = self._augment(x)
        # (M, T, V, C) → (C, T, V, M)
        x = x.permute(3, 1, 2, 0).contiguous()
        return x, self.y[idx]

    def _augment(self, x):
        # Rastgele rotasyon (±30 derece)
        angle = (torch.rand(1).item() - 0.5) * 60 * (3.14159 / 180)
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        R = torch.tensor([[cos_a, -sin_a], [sin_a, cos_a]], dtype=torch.float32)
        x = torch.einsum('mtvc,cd->mtvd', x, R)
        # Rastgele ölçekleme (0.9–1.1)
        scale = 0.9 + torch.rand(1).item() * 0.2
        x = x * scale
        return x

In [ ]:
print("Veri yükleniyor...")
d = np.load(DATA_PATH)
X_train, y_train = d['X_train'], d['y_train']
X_val,   y_val   = d['X_val'],   d['y_val']

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}    y_val  : {y_val.shape}")

BATCH_SIZE = 64

train_ds = NTUDataset(X_train, y_train, augment=True)
val_ds   = NTUDataset(X_val,   y_val,   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")

# Şekil kontrolü
xb, yb = next(iter(train_loader))
print(f"\nBatch shape: {xb.shape}  →  (N, C, T, V, M)")

## 2. ST-GCN Model

COCO 17-joint graph üzerinde Spatial Temporal GCN.
- 10 ST-GCN bloğu (spatial GCN + temporal conv + residual)
- Input: `(N, C=2, T=100, V=17, M=2)`

In [ ]:
# COCO 17-joint adjacency matrisini oluştur
COCO_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
NUM_JOINTS = 17

def build_adj(num_joints, pairs):
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    for i, j in pairs:
        A[i, j] = A[j, i] = 1.0
    A += np.eye(num_joints, dtype=np.float32)  # self-loop
    # Simetrik normalize: D^{-1/2} A D^{-1/2}
    D_inv_sqrt = np.diag(A.sum(1) ** -0.5)
    return torch.FloatTensor(D_inv_sqrt @ A @ D_inv_sqrt)

A = build_adj(NUM_JOINTS, COCO_PAIRS)
print(f"Adjacency matrix shape: {A.shape}")
print(f"Non-zero entries: {(A > 0).sum().item()}")

In [ ]:
class SpatialGCN(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.bn   = nn.BatchNorm2d(out_c)

    def forward(self, x):
        # x: (N, C, T, V)
        x = torch.einsum('nctv,vw->nctw', x, self.A)
        return self.bn(self.conv(x))


class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.0):
        super().__init__()
        self.gcn = SpatialGCN(in_c, out_c, A)
        pad = 4  # (9-1)//2
        self.tcn = nn.Sequential(
            nn.Conv2d(out_c, out_c, (9, 1), stride=(stride, 1), padding=(pad, 0)),
            nn.BatchNorm2d(out_c),
            nn.Dropout(dropout),
        )
        if in_c != out_c or stride != 1:
            self.residual = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=(stride, 1)),
                nn.BatchNorm2d(out_c)
            )
        else:
            self.residual = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = self.residual(x)
        x = self.relu(self.gcn(x))
        x = self.tcn(x)
        return self.relu(x + res)


class STGCN(nn.Module):
    def __init__(self, num_classes=60, in_channels=2, num_joints=17, A=None, dropout=0.5):
        super().__init__()
        A = A if A is not None else torch.eye(num_joints)

        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)

        # 10 blok: orijinal ST-GCN mimarisi
        self.layers = nn.ModuleList([
            STGCNBlock(in_channels, 64,  A),
            STGCNBlock(64,  64,  A),
            STGCNBlock(64,  64,  A),
            STGCNBlock(64,  64,  A),
            STGCNBlock(64,  128, A, stride=2),
            STGCNBlock(128, 128, A),
            STGCNBlock(128, 128, A),
            STGCNBlock(128, 256, A, stride=2),
            STGCNBlock(256, 256, A),
            STGCNBlock(256, 256, A),
        ])

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(256, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # x: (N, C, T, V, M)
        N, C, T, V, M = x.shape

        # Data BN: (N*M, C*V, T)
        x = x.permute(0, 4, 1, 3, 2).contiguous()  # (N, M, C, V, T)
        x = x.view(N * M, C * V, T)
        x = self.data_bn(x)
        x = x.view(N * M, C, V, T).permute(0, 1, 3, 2)  # (N*M, C, T, V)

        for layer in self.layers:
            x = layer(x)

        # (N*M, 256, T', V') → global pool → (N*M, 256)
        x = self.pool(x).view(N * M, -1)

        # Kişileri ortala: (N, M, 256) → (N, 256)
        x = x.view(N, M, -1).mean(dim=1)

        return self.fc(self.drop(x))


# Model boyutu kontrolü
model = STGCN(num_classes=60, in_channels=2, num_joints=17, A=A).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total_params:,}  ({total_params/1e6:.2f}M)")

# Forward pass testi
with torch.no_grad():
    dummy = torch.randn(4, 2, 100, 17, 2).to(device)
    out = model(dummy)
    print(f"Test forward pass: {dummy.shape} → {out.shape}")

## 3. Eğitim

In [ ]:
# Hyperparameter'lar
NUM_EPOCHS   = 80
LR           = 0.1
WEIGHT_DECAY = 5e-4
MOMENTUM     = 0.9

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True
)

# Cosine annealing: smooth LR düşüşü
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-4
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = GradScaler()  # Mixed precision için

print(f"Optimizer : SGD (lr={LR}, momentum={MOMENTUM}, wd={WEIGHT_DECAY})")
print(f"Scheduler : CosineAnnealing (T_max={NUM_EPOCHS})")
print(f"Loss      : CrossEntropy (label_smoothing=0.1)")
print(f"Epochs    : {NUM_EPOCHS}")

In [ ]:
def run_epoch(model, loader, optimizer, criterion, scaler, training):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    logits = model(x)
                    loss = criterion(logits, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                with autocast():
                    logits = model(x)
                    loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)

    return total_loss / total, correct / total

In [ ]:
history = defaultdict(list)
best_val_acc = 0.0
best_ckpt = os.path.join(OUT_DIR, 'best_model.pth')

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}  {'LR':>8}  {'Time':>6}")
print("-" * 72)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, criterion, scaler, training=True)
    va_loss, va_acc = run_epoch(model, val_loader,   optimizer, criterion, scaler, training=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    elapsed = time.time() - t0
    current_lr = scheduler.get_last_lr()[0]

    print(f"{epoch:5d}  {tr_loss:10.4f}  {tr_acc*100:8.2f}%  {va_loss:8.4f}  {va_acc*100:6.2f}%  {current_lr:.2e}  {elapsed:5.1f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_acc': va_acc, 'optimizer': optimizer.state_dict()}, best_ckpt)
        print(f"        ↑ En iyi model kaydedildi (val_acc={va_acc*100:.2f}%)")

    # Her 10 epoch'ta checkpoint
    if epoch % 10 == 0:
        ckpt_path = os.path.join(OUT_DIR, f'checkpoint_ep{epoch:03d}.pth')
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'history': dict(history)}, ckpt_path)

print(f"\nEğitim tamamlandı. En iyi val acc: {best_val_acc*100:.2f}%")

## 4. Sonuçlar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train')
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Val')
axes[1].axhline(best_val_acc*100, color='red', linestyle='--',
                label=f'Best Val={best_val_acc*100:.2f}%')
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('ST-GCN — NTU RGB+D 60 XSub', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=120)
plt.show()
print(f"En iyi val accuracy: {best_val_acc*100:.2f}%")

## 5. Confusion Matrix

In [ ]:
# En iyi modeli yükle
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model'])
print(f"Epoch {ckpt['epoch']} modeli yüklendi (val_acc={ckpt['val_acc']*100:.2f}%)")

# Tüm val tahminlerini topla
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        with autocast():
            logits = model(x)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(y)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

top1 = (all_preds == all_labels).mean() * 100
print(f"Val Top-1 Accuracy: {top1:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

# Normalize (per-class recall)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(18, 16))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_xticks(range(60))
ax.set_yticks(range(60))
short_labels = [f"{i}:{n[:12]}" for i, n in enumerate(NTU60_CLASSES)]
ax.set_xticklabels(short_labels, rotation=90, fontsize=6)
ax.set_yticklabels(short_labels, fontsize=6)
ax.set_xlabel('Tahmin Edilen')
ax.set_ylabel('Gerçek')
ax.set_title(f'Confusion Matrix (Normalized) — Val Acc: {top1:.2f}%')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# En çok karıştırılan 10 action pair
print("En çok karıştırılan action pair'ler (off-diagonal):")
print(f"{'Gerçek':40s}  {'Tahmin':40s}  {'Sayı':>5}")
print("-" * 90)

cm_copy = cm.copy()
np.fill_diagonal(cm_copy, 0)

pairs = []
for i in range(60):
    for j in range(60):
        if cm_copy[i, j] > 0:
            pairs.append((cm_copy[i, j], i, j))

for count, true_cls, pred_cls in sorted(pairs, reverse=True)[:10]:
    print(f"{NTU60_CLASSES[true_cls]:40s}  {NTU60_CLASSES[pred_cls]:40s}  {int(count):5d}")

In [ ]:
# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)

print("En düşük accuracy'li 10 class:")
worst = np.argsort(per_class_acc)[:10]
for i in worst:
    print(f"  [{i:2d}] {NTU60_CLASSES[i]:40s}: {per_class_acc[i]*100:.1f}%")

print("\nEn yüksek accuracy'li 10 class:")
best = np.argsort(per_class_acc)[-10:][::-1]
for i in best:
    print(f"  [{i:2d}] {NTU60_CLASSES[i]:40s}: {per_class_acc[i]*100:.1f}%")

## 6. Sonuç Özeti

| Metrik | Değer |
|--------|-------|
| Val Top-1 Accuracy | — |
| Literatür (ST-GCN, NTU60-XSub) | ~81% (3D Kinect) |
| Literatür (PYSKL ST-GCN, 2D HRNet) | ~85% |

**Sonraki adım:** `03_ablation.ipynb` — normalizasyon, temporal sampling, ST-GCN vs ST-GCN++ karşılaştırması.